# 2. Diamonds: recovering volume from raw dimensions

A larger table (~54,000 rows) and a quantity that is not any single column.

**Data:** `diamonds` — 53,940 stones with price, carat weight, cut grade and three
dimensions in mm.

**Target:** carat (weight) from `x`, `y`, `z`, `depth`, `table`. Weight is density x
volume, and volume is a three-way product no linear model can express.


## Setup


In [ ]:
%pip install -q "beamfeat[units]" pandas matplotlib seaborn scikit-learn


In [ ]:
import warnings, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import RidgeCV
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

from beamfeat import BeamFeatRegressor

warnings.filterwarnings("ignore", message=".*valid feature names.*")
sns.set_theme(style="whitegrid")
pd.set_option("display.width", 120)

SEED = 0


In [ ]:
from pathlib import Path
import urllib.request, urllib.error
import pandas as pd

CSV_DIR = Path("csv")
SEABORN = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/"


def load_csv(name, url=None, **read_kw):
    """Read csv/<name>, downloading it once if it isn't there yet."""
    CSV_DIR.mkdir(exist_ok=True)
    path = CSV_DIR / name

    if path.exists():
        print(f"cached   {path}  ({path.stat().st_size / 1e3:.0f} kB)")
    else:
        src = url or SEABORN + name
        print(f"fetching {name} ...", end=" ", flush=True)
        try:
            urllib.request.urlretrieve(src, path)
        except urllib.error.URLError as err:
            raise RuntimeError(f"download failed: {src}\n{err}") from None
        print(f"saved to {path}  ({path.stat().st_size / 1e3:.0f} kB)")

    return pd.read_csv(path, **read_kw)


In [ ]:
raw = load_csv("diamonds.csv")
print(raw.shape)
raw.head()


### Look before you fit

Three checks, every time:

- **shape and dtypes** — is anything a string that should be a number?
- **missing values** — `beamfeat` raises on `NaN` rather than imputing
- **variance** — any column below `1e-10` is dropped as constant, silently


In [ ]:
print("dtypes:")
print(raw.dtypes)
print("\nmissing:", int(raw.isna().sum().sum()))

raw[["carat", "depth", "table", "price", "x", "y", "z"]].describe().T


Look at the `min` row: `x`, `y` and `z` all bottom out at 0. A diamond with a
dimension of exactly zero does not exist — these are recording failures. The `max`
row shows the other problem: a `y` of 58.9 mm on a stone whose `x` is 10 mm.


In [ ]:
before = len(raw)
df = raw[(raw.x > 0) & (raw.y > 0) & (raw.z > 0) & (raw.y < 20) & (raw.z < 20)].copy()
print(f"dropped {before - len(df)} impossible rows; {len(df):,} remain")


In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(13, 3.6))

sns.heatmap(df[["x", "y", "z", "depth", "table", "carat"]].corr(),
            annot=True, fmt=".2f", cmap="RdBu_r", center=0, square=True,
            cbar=False, ax=ax[0])
ax[0].set_title("Correlations")

ax[1].scatter(df.x, df.carat, s=2, alpha=.1)
ax[1].set_xlabel("x (mm)"); ax[1].set_ylabel("carat")
ax[1].set_title("carat vs one dimension — curved")

ax[2].scatter(df.x * df.y * df.z, df.carat, s=2, alpha=.1)
ax[2].set_xlabel("x*y*z (mm$^3$)"); ax[2].set_ylabel("carat")
ax[2].set_title("carat vs volume — straight")

plt.tight_layout()
plt.show()


The middle panel curves; the right panel is a line. That difference *is* the feature
we want the search to find on its own.


## Does it scale?

Search cost is driven by depth and beam width, not row count. Measure rather than
assume.


In [ ]:
X = df[["x", "y", "z", "depth", "table"]].values
y = df["carat"].values
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=SEED)

for n in [5_000, 20_000, len(X_tr)]:
    t0 = time.time()
    m = BeamFeatRegressor(max_depth=2, beam_width=30,
                          random_state=SEED).fit(X_tr[:n], y_tr[:n])
    print(f"n = {n:>6,}   fit {time.time() - t0:5.1f}s   "
          f"R2 = {m.score(X_te, y_te):.5f}   {len(m.formulas())} feature(s)")


## The discovered feature


In [ ]:
model = BeamFeatRegressor(max_depth=2, beam_width=30, random_state=SEED).fit(X_tr, y_tr)

print("features :", model.formulas())
print("equation :", model.equation())
print("test R^2 :", round(model.score(X_te, y_te), 5))
print("FDR ok   :", model.fdr_controlled_)


A three-way product of dimensions — a volume in mm³.

You will often see `x * x * z` rather than `x * y * z`. Not an error: for round
brilliant cuts `x` and `y` are two perpendicular widths of the same circular girdle,
so they are nearly identical and the search picks whichever it reaches first.


In [ ]:
coef = (model.coef_ / model.scaler_.scale_)[0]
solid_box = 0.00352 / 0.2          # carat per mm^3 for a solid block of diamond

print(f"fitted coefficient   : {coef:.5f} carat/mm^3")
print(f"solid-block value    : {solid_box:.5f} carat/mm^3")
print(f"implied fill fraction: {coef / solid_box:.1%}")


About a third. A brilliant-cut stone is two cones joined at the girdle, so it fills
roughly 30–40% of its bounding box. The constant is physically sensible, which is a
stronger check than R².


## Baselines


In [ ]:
ridge = make_pipeline(StandardScaler(), RidgeCV()).fit(X_tr, y_tr)
gbm = HistGradientBoostingRegressor(random_state=SEED).fit(X_tr, y_tr)

print(f"ridge (raw)      R2 = {ridge.score(X_te, y_te):.5f}")
print(f"gradient boost   R2 = {gbm.score(X_te, y_te):.5f}")
print(f"beamfeat         R2 = {model.score(X_te, y_te):.5f}  "
      f"({len(model.formulas())} feature)")


## Target transforms matter more than the search

Same inputs, harder target: predict **price**. Price is roughly exponential in size,
so the natural move is to model `log(price)` instead.


In [ ]:
Xp = df[["carat", "x", "y", "z", "depth", "table"]].values
yp = df["price"].values
Ptr, Pte, qtr, qte = train_test_split(Xp, yp, test_size=0.3, random_state=SEED)

m_raw = BeamFeatRegressor(max_depth=2, beam_width=30, random_state=SEED).fit(Ptr, qtr)
m_log = BeamFeatRegressor(max_depth=2, beam_width=30,
                          random_state=SEED).fit(Ptr, np.log(qtr))

print(f"price       R2 = {m_raw.score(Pte, qte):.4f}   "
      f"({len(m_raw.formulas())} features)")
print(f"log(price)  R2 = {m_log.score(Pte, np.log(qte)):.4f}   "
      f"({len(m_log.formulas())} features)")
print("\nlog(price) formulas:", m_log.formulas())


Modelling the log gains about 0.07 R² and needs fewer features. No change to the
search — only to the question.

Two cautions. The two R² values are computed on **different scales** and are not
directly comparable; to compare fairly, back-transform and score on the original
units. And if you back-transform predictions with `exp`, remember that
$\mathbb{E}[\exp(z)] \neq \exp(\mathbb{E}[z])$, so the result is biased low unless
you correct for it.


## Takeaways

1. Recovered volume — a three-way product — from raw dimensions, at ~54,000 rows.
2. Fit time is flat in row count over this range; depth and beam width drive cost.
3. The fitted constant implies a ~35% fill fraction, which matches the cut geometry.
4. Ridge cannot represent a three-way interaction at all. This is where construction
   earns its place, unlike notebook 1.
5. Choosing the right target transform beat anything the search could do.
